Galaxies
========

This tutorial shows how to use galaxies, including visualizing and extracting their individual light profiles.

__Contents__

- **JAX:** Jitting galaxy calculations yourself, and the pytree registration it needs — see `scripts/guides/using_jax.py`.
- **Units:** The internal unit system used throughout this example.
- **Data Structures:** Overview of bespoke data structures and the slim/native API.
- **Grids:** Create a 2D grid of (y,x) coordinates for evaluating galaxy light.
- **Light Profiles:** Define Sersic and other light profiles to describe a galaxy's luminosity distribution.
- **Galaxies:** Create Galaxy objects from multiple light profiles and compute their images.
- **Galaxies (multiple):** Group multiple galaxies into a Galaxies object and compute combined images.
- **Log10:** Plot galaxy images in log10 space for clearer visualization of outskirts.
- **Extending Objects:** Create complex multi-galaxy systems with many light profiles.
- **Data Structures Slim / Native:** Access images in 1D slim or 2D native representations.
- **Individual Galaxy Components:** Extract and plot individual light profile components from galaxies.
- **Galaxies Composition:** Summarize the composition of galaxies and their profiles.
- **One Dimension Projection:** Create projected 2D radial grids for 1D profile calculations.
- **One Dimensional Quantities:** Compute and plot 1D radial profiles of galaxy light.
- **Decomposed 1D Plot:** Plot decomposed 1D profiles showing each light component separately.
- **Modeling Results:** Pointers to results-specific functionality for galaxy models.

__JAX__

Galaxy calculations are JAX-accelerated automatically inside a model-fit. To JIT them yourself — including the
pytree registration a jitted `Galaxy` argument needs — see the executable recipes in
`scripts/guides/using_jax.py`.

__Units__

In this example, all quantities are **PyAutoGalaxy**'s internal unit coordinates, with spatial coordinates in
arc seconds, luminosities in electrons per second and mass quantities (e.g. convergence) are dimensionless.

The guide `guides/units/cosmology.ipynb` illustrates how to convert these quantities to physical units like
kiloparsecs, magnitudes and solar masses.

__Data Structures__

Quantities inspected in this example script use **PyAutoGalaxy** bespoke data structures for storing arrays, grids,
vectors and other 1D and 2D quantities. These use the `slim` and `native` API to toggle between representing the
data in 1D numpy arrays or high dimension numpy arrays.

This tutorial will only use the `slim` properties which show results in 1D numpy arrays of
shape [total_unmasked_pixels]. This is a slimmed-down representation of the data in 1D that contains only the
unmasked data points

These are documented fully in the `autogalaxy_workspace/*/guides/data_structures.ipynb` guide.

__Google Colab Setup__

This cell sets up the environment when the notebook is run on Google Colab: it installs the
required PyAuto packages, clones the workspace (configuration files and example datasets) and
points the configuration at it. If you are running the notebook elsewhere (e.g. locally via
your own installation) it does nothing, and you can run it safely.

Colab tip: model-fits run much faster on a GPU — enable one via "Runtime" -> "Change runtime
type" -> "Hardware accelerator" before running the notebook.

In [ ]:
try:
    import google.colab
except ImportError:
    from autogalaxy import setup_colab as _setup_colab
else:
    import importlib
    import subprocess
    import sys

    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "autonerves", "--no-deps"]
    )
    _setup_colab = importlib.import_module("autonerves.setup_colab")

_setup_colab.setup("autogalaxy")

In [ ]:

from autogalaxy import setup_notebook; setup_notebook()

import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
import autofit as af
import autogalaxy as ag
import autogalaxy.plot as aplt


__Grids__

To describe the luminous emission of galaxies, **PyAutoGalaxy** uses `Grid2D` data structures, which are 
two-dimensional Cartesian grids of (y,x) coordinates. 

Below, we make and plot a uniform Cartesian grid:

In [ ]:
grid = ag.Grid2D.uniform(
    shape_native=(100, 100),
    pixel_scales=0.1,  # The pixel-scale describes the conversion from pixel units to arc-seconds.
)

aplt.plot_grid(grid=grid, title="Grid")

__Light Profiles__

We will use this `Grid2D`'s coordinates to evaluate the galaxy's morphology. We therefore need analytic 
functions representing a galaxy's light distribution(s). 

For this,  **PyAutoGalaxy** uses `LightProfile` objects, for example the `Sersic` `LightProfile` object which
represents a light distribution:

In [ ]:
sersic_light_profile = ag.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=(0.2, 0.1),
    intensity=0.005,
    effective_radius=2.0,
    sersic_index=4.0,
)

By passing this profile a `Grid2D`, we evaluate the light at every (y,x) coordinate on the `Grid2D` and create an 
image of the `LightProfile`.

In [ ]:
image = sersic_light_profile.image_2d_from(grid=grid)

The PyAutoGalaxy plot module provides methods for plotting objects and their properties, like 
the `LightProfile`'s image.

In [ ]:
aplt.plot_array(array=sersic_light_profile.image_2d_from(grid=grid), title="Image")

__Galaxies__

A `Galaxy` object is a collection of `LightProfile` objects at a given redshift. 

The code below creates a galaxy which is made of two components, a bulge and disk.

In [ ]:
bulge = ag.lp.Sersic(
    centre=(0.0, 0.0),
    ell_comps=ag.convert.ell_comps_from(axis_ratio=0.9, angle=45.0),
    intensity=1.0,
    effective_radius=0.6,
    sersic_index=3.0,
)

disk = ag.lp.Exponential(
    centre=(0.0, 0.0),
    ell_comps=ag.convert.ell_comps_from(axis_ratio=0.7, angle=30.0),
    intensity=0.5,
    effective_radius=1.6,
)

galaxy = ag.Galaxy(redshift=0.5, bulge=bulge, disk=disk)

We can create an image the galaxy by passing it the 2D grid above.

In [ ]:
image = galaxy.image_2d_from(grid=grid)

The **PyAutoGalaxy** plot module provides methods for plotting galaxies.

Below, we plot its image, which is the sum of the bulge and disk components.

In [ ]:
aplt.plot_array(array=galaxy.image_2d_from(grid=grid), title="Image")

__Galaxies__

If our observation contains multiple galaxies, we can create a `Galaxies` object to represent all galaxies.

By passing `Galaxy` objects to a `Galaxies`, **PyAutoGalaxy** groups them to indicate they are at the same redshift.

In [ ]:
galaxy_0 = ag.Galaxy(
    redshift=0.5,
    bulge=ag.lp.Sersic(
        centre=(0.0, -1.0),
        ell_comps=(0.25, 0.1),
        intensity=0.1,
        effective_radius=0.8,
        sersic_index=2.5,
    ),
)

galaxy_1 = ag.Galaxy(
    redshift=0.5,
    bulge=ag.lp.Sersic(
        centre=(0.0, 1.0),
        ell_comps=(0.0, 0.1),
        intensity=0.1,
        effective_radius=0.6,
        sersic_index=3.0,
    ),
)

galaxies = ag.Galaxies(galaxies=[galaxy_0, galaxy_1])

The image of all galaxies summed can easily be computed from this object.

**PyAutoGalaxy** plot tools allow us to plot this image or a subplot containing images of each individual galaxy.

In [ ]:
image = galaxies.image_2d_from(grid=grid)

aplt.plot_array(array=galaxies.image_2d_from(grid=grid), title="Image")
aplt.subplot_galaxies(galaxies=galaxies, grid=grid)

__Log10__

The light distributions of galaxies are closer to a log10 distribution than a linear one. 

This means that when we plot an image of a light profile, its appearance is better highlighted when we take the
logarithm of its values and plot it in log10 space.

The `plot_array` function has an input `use_log10`, which does this automatically.
Below, we can see that the image plotted now appears more clearly, with the outskirts of the light profile more visible.

In [ ]:
aplt.plot_array(array=galaxies.image_2d_from(grid=grid), title="Image", use_log10=True)

__Extending Objects__

The PyAutoGalaxy API is designed such that all of the objects introduced above are extensible. `Galaxy` objects 
can take many `LightProfile`'s and `Galaxies`'s many `Galaxy`'s. 

To finish, lets create 2 merging galaxies, where the second galaxy has multiple star forming clumps.

In [ ]:
galaxy_0 = ag.Galaxy(
    redshift=0.5,
    bulge=ag.lmp.Sersic(
        centre=(0.0, 0.0),
        ell_comps=(0.0, 0.05),
        intensity=0.5,
        effective_radius=0.3,
        sersic_index=3.5,
        mass_to_light_ratio=0.6,
    ),
    disk=ag.lmp.Exponential(
        centre=(0.0, 0.0),
        ell_comps=(0.0, 0.1),
        intensity=1.0,
        effective_radius=2.0,
        mass_to_light_ratio=0.2,
    ),
)

galaxy_1 = ag.Galaxy(
    redshift=1.0,
    bulge=ag.lp.Exponential(
        centre=(0.00, 0.00),
        ell_comps=(0.05, 0.05),
        intensity=1.2,
        effective_radius=0.1,
    ),
    extra_galaxy_0=ag.lp.Sersic(centre=(1.0, 1.0), intensity=0.5, effective_radius=0.2),
    extra_galaxy_1=ag.lp.Sersic(centre=(0.5, 0.8), intensity=0.5, effective_radius=0.2),
    extra_galaxy_2=ag.lp.Sersic(
        centre=(-1.0, -0.7), intensity=0.5, effective_radius=0.2
    ),
)

galaxies = ag.Galaxies(galaxies=[galaxy_0, galaxy_1])

This is what the merging galaxies look like:

In [ ]:
aplt.plot_array(array=galaxies.image_2d_from(grid=grid), title="Image")

__Data Structures Slim / Native__

The images above are returned as a 1D numpy array. 

**PyAutoLens** includes dedicated functionality for manipulating this array, for example mapping it to 2D or
performing the calculation on a high resolution sub-grid which is then binned up. 

This uses the data structure API, which is described in the `guides/data_structures.py` example. This 
tutorial will avoid using this API, but if you need to manipulate results in more detail you should check it out.

In [ ]:
print(image.slim)

__Individual Galaxy Components__

We are able to create an image of each galaxy as follows, which includes the emission of only one galaxy at a
time.

In [ ]:
image = galaxies[0].image_2d_from(grid=grid)
image = galaxies[1].image_2d_from(grid=grid)

In order to create images of each light profile (e.g. the `bulge`), we can extract each individual component from 
each galaxy.

The list of galaxies is in order of how we specify them in the `collection` above.

In [ ]:
bulge_0 = galaxies[0].bulge
bulge_1 = galaxies[1].bulge

You could easily extract the `disk`  of `galaxy_0`:

 disk_0 = galaxies[0].disk

Finally, we can use the extracted bulge components to make images of the bulge.

In [ ]:
bulge_0_image_2d = bulge_0.image_2d_from(grid=grid)
bulge_1_image_2d = bulge_1.image_2d_from(grid=grid)

print(bulge_0_image_2d.slim[0])
print(bulge_1_image_2d.slim[0])

It is more concise to extract these quantities in one line of Python:

In [ ]:
bulge_0_image_2d = galaxies[0].bulge.image_2d_from(grid=grid)

It is straight forward to extract and plot an individual light profile component.

In [ ]:
aplt.plot_array(array=galaxies[0].bulge.image_2d_from(grid=grid), title="Image")

__Galaxies__

We extracted the `bulge` light profiles of each galaxy. 

We can just as easily extract each `Galaxy` and use it to perform the calculations above.

In [ ]:
galaxy_0 = galaxies[0]

galaxy_0_image_2d = galaxy_0.image_2d_from(grid=grid)

We can also plot the galaxy, for example a subplot of each individual light profile 
image (which because this galxy is only a single bulge, is a single image).

In [ ]:
aplt.plot_array(array=galaxy_0.image_2d_from(grid=grid), title="Image")

__Galaxies Composition__

Lets quickly summarise what we've learnt by printing every object:

In [ ]:
print(galaxies)
print(galaxies[0])
print(galaxies[0])
print(galaxies[0].bulge)
print(galaxies[1].bulge)
print()

__One Dimension Projection__

We often want to calculative 1D quantities of a light profile, for example to plot how its light changes as
a function of radius.

To do this, we must still input a 2D grid into the `image_2d_from` method, therefore we create a project 2D 
radial grid as follows which has shape [Number_of_1d_coordinates, 2] and where all [:,0] entries are the same.

A simple example of such a grid is as follows with 4 1D coordinates is:

In [ ]:
grid_2d_projected = ag.Grid2DIrregular(
    [
        [1.000000e-06, 1.000000e-06],
        [1.000000e-06, 1.000001e00],
        [1.000000e-06, 2.000001e00],
        [1.000000e-06, 3.000001e00],
    ]
)

As in this example, we often already have a 2D grid we are using to calculate images of a ligth profile
and it would be convenient to simply create `grid_2d_projected` from that.

For example, we may want the project grid which traces it major axis in uniform radial steps.

This is easily computed using the `grid_2d_radial_project_from` function and passing the `centre` and `angle`
of a light profile we can make it align with the light profile itself.

Note how in this example the two galaxy bulges are not rotationally aligned but we aligned the projected
grid with the first galaxy. The centres are aligned, but if they were not that would cause similar
issues.

In [ ]:
grid_2d_projected = grid.grid_2d_radial_projected_from(
    centre=galaxy_0.bulge.centre, angle=galaxy_0.bulge.angle()
)

__One Dimensional Quantities__

We can now compute quantities quantities in 1D using the project grid, for inspection and visualization.

To do this, we perform the following steps:

For example, from a light profile or galaxy we can compute its `image_1d`, which provides us with its image values
(e.g. luminosity) as a function of radius.

In [ ]:
galaxy_0 = galaxies[0]
image_1d = galaxy_0.image_2d_from(grid=grid_2d_projected)
print(image_1d)

galaxy_1 = galaxies[1]
image_1d = galaxy_1.image_2d_from(grid=grid_2d_projected)
print(image_1d)

When we plot 1D quantities, we do not use built-in plotting functions as in 2D, but instead use standard
matplotlib functionality.

The reason is partly that 1D plotting is simple, but also because 1D plots have many different decisions
about what is plotted and how they are computed, meaning its better to give the user full control.

In [ ]:
plt.plot(grid_2d_projected[:, 1], image_1d)
plt.xlabel("Radius (arcseconds)")
plt.ylabel("Luminosity")
plt.show()
plt.close()

If we want a specific 1D radial profile over a certain range of coordinates, we create a `Grid2DIrregular` of
(y,x) coordinates along the x-axis and evaluate the image on it.

In [ ]:
radii = np.arange(10000) * 0.01
grid_radial = ag.Grid2DIrregular(values=[(0.0, r) for r in radii])
image_1d = galaxy_0.image_2d_from(grid=grid_radial)

plt.plot(radii, image_1d)
plt.xlabel("Radius (arcseconds)")
plt.ylabel("Luminosity")
plt.show()
plt.close()

__Decomposed 1D Plot__

We can make calculate a plot containing every individual light profile of a galaxy in 1D, for example showing a  
decomposition of its `bulge` and `disk`.

Every profile on a decomposed plot is computed using a radial grid centred on its profile centre and aligned with
its major-axis. Therefore 2D offsets between the components are not portray in such a figure.

In [ ]:
grid_2d_projected = grid.grid_2d_radial_projected_from(
    centre=galaxy_0.bulge.centre, angle=galaxy_0.bulge.angle()
)
bulge_image_1d = galaxy_0.bulge.image_2d_from(grid=grid_2d_projected)

grid_2d_projected = grid.grid_2d_radial_projected_from(
    centre=galaxy_0.disk.centre, angle=galaxy_0.disk.angle()
)
disk_image_1d = galaxy_0.disk.image_2d_from(grid=grid_2d_projected)

plt.plot(grid_2d_projected[:, 1], bulge_image_1d, label="Bulge")
plt.plot(grid_2d_projected[:, 1], disk_image_1d, label="Disk")
plt.xlabel("Radius (arcseconds)")
plt.ylabel("Luminosity")
plt.legend()
plt.show()
plt.close()

__Modeling Results__

Modeling uses a non-linear search to fit a model of galaxies to a dataset.

It is illustrated in the `modeling` packages of `autogalaxy_workspace`.

Modeling results have some specific functionality and use cases, which are described in the `results` packages of
`autogalaxy_workspace`,  in particular the `galaxies_fit.py` example script which describes: 

 - `Max Likelihood`: Extract and plot the galaxy models which maximize the likelihood of the fit.
 - `Samples`, Extract the samples of the non-linear search and inspect specific parameter values.
 - `Errors`: Makes plots that quantify the errors on the inferred galaxy properties.
 - `Refitting` Refit specific models from the modeling process to the dataset. 

__Wrap Up__

This tutorial explained how to compute the results of an inferred model from a galaxies.

We have learnt how to extract individual galaxies and light profiles from the results of
a model-fit and use these objects to compute specific quantities of each component.